# Peak calling for specific cases

Using the latest on September 29th, 2025

** ALSO, will include more normal cells (CN > 3) instead of cutting off at 5 to capture 
low concentrating ones **

Run the peak calling algorithm, annotate the peaks, save the number peaks, then plot the peaks. 


1. Load the list of data
2. For each data:
    1. load the data
    2. filter for individual-gene from the summary file
    3. in each file, filter cells to only tumors from the qc files
    4. run the peak caller
    5. save the results
    6. plot the annotated peaks
5. Gather the results in one place


Use the same file that is used to run the main mixture model pipeline.

In [ ]:
from peak_detection_utils import handle_item, get_paths, get_tumor_cells, setup_dirs
import os
import pandas as pd
import matplotlib.pyplot as plt
from glob import glob

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

# Setup Original Oncogene CNV files
oncogene_files = glob(
    "oncogene_cn_final//*.sv-seg-cn.oncogene.cn.csv"
)

# Load the ecDNA mixture model results
oncogenes = pd.read_csv(
    "/tables/processed_model_results_nov_07_2025.csv"
)

# Setup the output directories
outDir = "."

# Minimum process
figuresDir, tablesDir, dataDir = setup_dirs(outDir)

oncogenes["individual"] = oncogenes["system_name"].tolist()
oncogenes["gene"] = oncogenes["locus"].tolist()
oncogenes["id"] = (
    oncogenes["individual"].astype(str) + "__" + oncogenes["gene"].astype(str)
)

ignore_qc = True
do_plot = False
results = None

minimal_normal = 3  # Include more normal cells with CN > 3

missed = []
for i, item in enumerate(oncogenes.iterrows()):
    item = item[1]
    individual = item["individual"]
    # Extract the cn path and qc path from oncogene_files
    try:
        fpath, qc_path = get_paths(
            individual=individual, ignore_qc=ignore_qc, oncogene_files=oncogene_files
        )
        locus = item["gene"]
        if ignore_qc:
            tumor_cells = None
        else:
            tumor_cells = get_tumor_cells(qc_path=qc_path)
        print(f"Processing {fpath} for locus {locus}")
        dataset_name, locus, n_peaks, pickle_path = handle_item(
            fpath=fpath,
            locus=locus,
            verbose=False,
            do_plot=do_plot,
            tumor_cells=tumor_cells,
            figuresDir=figuresDir,
            dataDir=dataDir,
            drop_normals=True,
            minimal_normal=minimal_normal,
        )
        # Create a tmp data.frame to add to the results
        tmp_df = pd.DataFrame(
            {
                "dataset_name": [dataset_name],
                "locus": [locus],
                "n_peaks": [n_peaks],
                "pickle_path": [pickle_path],
            }
        )
        if results is None:
            results = tmp_df
        else:
            results = pd.concat([results, tmp_df], ignore_index=True)
    except Exception as e:
        print(f"Error processing {individual}, {locus}: {e}")
        missed.append((individual, locus, str(e)))